In [1]:
# packages
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from sklearn.tree import \
      (DecisionTreeRegressor as DTR, 
       DecisionTreeClassifier as DTC, 
       plot_tree)
from sklearn.ensemble import \
     (RandomForestRegressor as RFR,
      RandomForestClassifier as RFC,
      GradientBoostingRegressor as GBR)
from sklearn.model_selection import cross_val_score

# set seed
seed = 5498

### You'll need the heart attack data (heart.csv) to complete this exercise. Your goal is to predict the risk of having a heart attack (binary).

In [3]:
heart = pd.read_csv('data/heart.csv')
heart.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/heart.csv'

In [ ]:
heart.dtypes

In [ ]:
heart.shape

### Feature engineering

In [ ]:
def feature_engineering(df=heart):
    df['Male'] = (df.Sex == 'Male').astype(int)
    df['HealthyDiet'] = (df.Diet == 'Healthy').astype(int)
    df['UnhealthyDiet'] = (df.Diet == 'Unhealthy').astype(int)
    df[['BP1','BP2']] = df['Blood Pressure'].str.split('/', n=1, expand=True).astype(int)

feature_engineering()

In [ ]:
heart.head()

In [ ]:
indep_vars = ['Age', 'Male', 'Cholesterol', 'BP1', 'BP2', 'Heart Rate', 'Diabetes', 
              'Family History', 'Smoking', 'Obesity', 'Alcohol Consumption', 'HealthyDiet', 
              'UnhealthyDiet', 'Exercise Hours Per Week', 'Previous Heart Problems', 
              'Medication Use', 'Stress Level', 'Sedentary Hours Per Day', 'Income', 'BMI',
              'Triglycerides', 'Physical Activity Days Per Week', 'Sleep Hours Per Day']

X = heart[indep_vars]
y = heart['Heart Attack Risk']

In [ ]:
X_train, X_test, y_train, y_test, Train, Test = train_test_split(X, y, heart, 
                                                                 random_state = seed, 
                                                                 test_size = 0.33, 
                                                                 shuffle = True)

### Univariate Analyses (Mean, Median, etc.)

In [ ]:
Train.describe()

### Correlation Matrix

In [ ]:
X_train.corr().style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

### To predict heart attack risk, we'll start by building a few simple decision trees.

In [ ]:
tree1 = DTC(max_depth=1)
tree1.fit(X_train, y_train)

tree2 = DTC(max_depth=2)
tree2.fit(X_train, y_train)

tree3 = DTC(max_depth=3)
tree3.fit(X_train, y_train)

### We can draw plots of each of these decision trees

In [ ]:
feature_names = list(X_train.columns)
ax = subplots(figsize=(6,6))[1]
plot_tree(tree1,
          feature_names=feature_names,
          ax=ax);

In [ ]:
ax = subplots(figsize=(12,6))[1]
plot_tree(tree2,
          feature_names=feature_names,
          ax=ax);

In [ ]:
ax = subplots(figsize=(18,8))[1]
plot_tree(tree3,
          feature_names=feature_names,
          ax=ax);

### Create a new tree with depth 3 that requires there to be at least 100 records in each leaf.

In [ ]:
tree4 = #fillin

ax = subplots(figsize=(18,8))[1]
plot_tree(tree4,
          feature_names=feature_names,
          ax=ax);

### Create a new version of the last tree so that it limits the number of leaf nodes to 8 rather than limiting the depth to 3.

In [ ]:
tree5 = #fillin

ax = subplots(figsize=(18,8))[1]
plot_tree(tree5,
          feature_names=feature_names,
          ax=ax);

### Create a basic random forest model on the training set.

In [ ]:
rf_basic = RFC(max_features=len(indep_vars),
               criterion='log_loss',
               n_estimators=50,
               max_depth=10,
               min_samples_leaf=1,
               bootstrap=True,
               oob_score=True,
               random_state=seed)
rf_basic.fit(X_train, y_train)

### We'll use this function to quickly calculate the log loss of a model.

In [ ]:
def log_loss(y_true, y_prob, eps=1e-15):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    
    # avoid log(0)
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    ll = -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    return ll

### Compute the log loss on the training set.

In [ ]:
y_hat_basic_train = rf_basic.predict_proba(X_train)[:,1]
print('train log loss:', log_loss(y_train, y_hat_basic_train))

### Calculate the out-of-bag error (bootstrapping) on training data to estimate the log loss on the test set.

In [ ]:
y_hat_basic_oob = rf_basic.oob_decision_function_[:,1]
print('oob log loss:  ', log_loss(y_train, y_hat_basic_oob))

### Compute the log loss on the test set.

In [ ]:
y_hat_basic_test = rf_basic.predict_proba(X_test)[:,1]
print('test log loss: ', log_loss(y_test, y_hat_basic_test))

### Build an improved random forest model on this same set of data. 

You will be scored according to how well it performs on an unseen holdout set.

In [ ]:
# Documentation: https://scikit-learn.org/dev/modules/generated/sklearn.ensemble.RandomForestRegressor.html

rf_better = #fillin

### Calculate the out-of-bag log loss for this model. Store it as `log_loss_oob`

In [ ]:
y_hat_better_oob = #fillin

log_loss_oob = #fillin
print(log_loss_oob)

### Create the feature importance list for this model.

In [ ]:
feature_imp = pd.DataFrame(
    {'importance':rf_better.feature_importances_},
    index=indep_vars)
feature_imp

### Sort the importance values and then create a horizontal bar chart that displays the sorted variable importances.

In [ ]:
feature_imp_sorted = #fillin

# Draw a horizontal bar chart
#fillin

## Discussion Questions

### (1) Explain what "value" means in the decision tree plots.

### (2) How is “log loss” different from the log likelihood we defined in class?

### (3) Explain how we know the basic model is overfit before attempting to build any other models.